In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'Ready in: {os.getcwd()}')


In [ ]:
import numpy as np
import pandas as pd
import json, time, joblib
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import spearmanr

SEED = 42
np.random.seed(SEED)

DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
MODEL_VARIANTS = ['5class_cw', '5class_smote']
CLASS_NAMES_5 = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
K_VALUES = [5, 10, 15]
K_HEADLINE = 10

# SHAP output spaces per model, as produced by 04c: sklearn RF TreeExplainer raw output is predict_proba,
# XGBoost TreeExplainer raw output is the per-class margin (log-odds), GradientExplainer on the softmax
# head is probability. The DNN logit head is the one new computation in this notebook; TreeExplainer
# cannot express multiclass softmax XGBoost in probability space, so the matched comparison is in logit space.
SOURCES = {
    'rf_prob':    ('rf',  'shap_shared'),
    'xgb_logit':  ('xgb', 'shap_shared'),
    'dnn_prob':   ('dnn', 'shap_shared'),
    'dnn_logit':  ('dnn', 'shap_shared_logit'),
}
# (label, source A, source B, note). RC takes top-k from A; both directions are computed below.
CONFIGS = [
    ('rf-xgb original',        'rf_prob',   'xgb_logit', 'prob vs logit, as in 06'),
    ('rf-dnn original',        'rf_prob',   'dnn_prob',  'prob vs prob, as in 06 (matched)'),
    ('xgb-dnn original',       'xgb_logit', 'dnn_prob',  'logit vs prob, as in 06 (mismatched)'),
    ('xgb-dnn logit-matched',  'xgb_logit', 'dnn_logit', 'logit vs logit'),
    ('rf-dnn logit-mismatched','rf_prob',   'dnn_logit', 'prob vs logit (mismatch control)'),
    ('dnn self prob-vs-logit', 'dnn_prob',  'dnn_logit', 'same model, two output heads'),
]

N_BOOTSTRAP = 10000
BOOTSTRAP_SEED = 42

TABLES = Path(REPO) / 'results' / 'tables'
DOCS = Path(REPO) / 'docs'
PREFIX = 'krishna_space'
print(f'{len(DATASETS)} datasets x {len(MODEL_VARIANTS)} variants x {len(CONFIGS)} configs x {len(K_VALUES)} K; B={N_BOOTSTRAP}')


In [ ]:
def find_dnn_path(dataset, model_name):
    p = Path(REPO) / 'models' / dataset / f'{model_name}.pt'
    if p.exists():
        return p
    raise FileNotFoundError(f'No DNN file at {p}')

def shap_mod():
    try:
        import shap
    except ImportError:
        os.system('pip install -q shap')
        import shap
    return shap

_torch = {}
def torch_env():
    if not _torch:
        import torch, torch.nn as nn
        _torch['torch'], _torch['nn'] = torch, nn
        _torch['DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
        torch.manual_seed(SEED)

        class DNN(nn.Module):
            def __init__(self, in_dim, n_classes, hidden=(256, 128, 64, 32), dropout=0.3):
                super().__init__()
                layers, prev = [], in_dim
                for h in hidden:
                    layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
                    prev = h
                layers.append(nn.Linear(prev, n_classes))
                self.net = nn.Sequential(*layers)
            def forward(self, x):
                return self.net(x)

        _torch['DNN'] = DNN
    return _torch

def load_dnn_raw(path):
    # Returns the network with its logit head (no softmax), eval mode.
    T = torch_env(); torch = T['torch']
    ckpt = torch.load(path, map_location=T['DEVICE'], weights_only=False)
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        in_dim, n_classes = ckpt['in_dim'], ckpt['n_classes']
        hidden, dropout, state_dict = tuple(ckpt['hidden']), ckpt['dropout'], ckpt['state_dict']
    else:
        state_dict = ckpt
        in_dim = state_dict['net.0.weight'].shape[1]
        n_classes = state_dict['net.16.weight'].shape[0]
        hidden, dropout = (256, 128, 64, 32), 0.3
    model = T['DNN'](in_dim=in_dim, n_classes=n_classes, hidden=hidden, dropout=dropout)
    model.load_state_dict(state_dict)
    return model.to(T['DEVICE']).eval()

def _to_nfc(shap_values, n_samples, n_features, n_classes=5):
    if isinstance(shap_values, list):
        shap_values = np.stack(shap_values, axis=-1)
    if shap_values.shape == (n_classes, n_samples, n_features):
        shap_values = np.transpose(shap_values, (1, 2, 0))
    elif shap_values.shape == (n_samples, n_classes, n_features):
        shap_values = np.transpose(shap_values, (0, 2, 1))
    assert shap_values.shape == (n_samples, n_features, n_classes), shap_values.shape
    return shap_values

def ensure_dnn_logit_shap(ds, model_name):
    # Identical to 04c compute_shap_canonical for the DNN (same eval_idx, same 50-sample background,
    # same GradientExplainer) except the explained output is the logit vector instead of softmax.
    out_path = Path(REPO) / 'shap_values' / ds / f'{model_name}_shap_shared_logit.npy'
    if out_path.exists():
        return out_path, True
    shap = shap_mod()
    T = torch_env(); torch = T['torch']
    proc = Path(REPO) / 'data' / 'processed' / ds
    eval_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy')
    bg_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_bg_idx.npy')
    X_eval = np.load(proc / 'X_test.npy').astype(np.float32)[eval_idx]
    X_bg = np.load(proc / 'X_calib.npy').astype(np.float32)[bg_idx]
    model = load_dnn_raw(find_dnn_path(ds, model_name))
    torch.manual_seed(SEED)
    bg_t = torch.from_numpy(X_bg).to(T['DEVICE'])
    ev_t = torch.from_numpy(X_eval).to(T['DEVICE'])
    sv = shap.GradientExplainer(model, bg_t).shap_values(ev_t)
    sv = _to_nfc(sv, len(eval_idx), X_eval.shape[1])
    np.save(out_path, sv.astype(np.float32))
    with open(Path(REPO) / 'shap_values' / ds / f'{model_name}_shap_shared_logit_meta.json', 'w') as f:
        json.dump({'model': model_name, 'dataset': ds, 'shap_target': 'logits (pre-softmax)',
                   'explainer': 'GradientExplainer', 'n_background': int(len(bg_idx)), 'n_eval': int(len(eval_idx)),
                   'seed': SEED, 'timestamp': datetime.now().isoformat()}, f, indent=2)
    del model, bg_t, ev_t
    if T['DEVICE'] == 'cuda':
        torch.cuda.empty_cache()
    return out_path, False

print('SHAP helpers ready')


In [ ]:
def feature_agreement(shap_a, shap_b, k):
    top_a = set(np.argsort(-np.abs(shap_a))[:k])
    top_b = set(np.argsort(-np.abs(shap_b))[:k])
    return len(top_a & top_b) / k

def rank_agreement(shap_a, shap_b, k):
    rank_a = np.argsort(-np.abs(shap_a))[:k]
    rank_b = np.argsort(-np.abs(shap_b))[:k]
    return sum(1 for i in range(k) if rank_a[i] == rank_b[i]) / k

def rank_correlation(shap_a, shap_b, k):
    # Spearman between the magnitude rankings, within A's top-k feature set, of A and of B (06 definition).
    top_a_idx = np.argsort(-np.abs(shap_a))[:k]
    ranks_a = np.argsort(np.argsort(-np.abs(shap_a)[top_a_idx])) + 1
    ranks_b = np.argsort(np.argsort(-np.abs(shap_b)[top_a_idx])) + 1
    if k < 2:
        return 0.0
    corr, _ = spearmanr(ranks_a, ranks_b)
    return float(corr) if not np.isnan(corr) else 0.0

def pairwise_rank_agreement(shap_a, shap_b, k):
    # For each ordered pair in A's top-k (A says fi >= fj), does B agree? Verbatim from 06.
    top_a = np.argsort(-np.abs(shap_a))[:k]
    abs_b = np.abs(shap_b)
    n_pairs, agree = 0, 0
    for i in range(k):
        for j in range(i + 1, k):
            fi, fj = top_a[i], top_a[j]
            n_pairs += 1
            if abs_b[fi] >= abs_b[fj]:
                agree += 1
    return agree / n_pairs if n_pairs > 0 else 0.0

METRICS = {'FA': feature_agreement, 'RA': rank_agreement, 'RC': rank_correlation, 'PRA': pairwise_rank_agreement}

def aggregate_vector(shap_arr, i):
    # |SHAP| summed over classes (06 aggregate mode). Sign agreement is undefined on this vector, so it is not reported.
    return np.abs(shap_arr[i]).sum(axis=-1)

def boot_index(n, B=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    return np.random.RandomState(seed).randint(0, n, size=(B, n))

def boot_weights(idx, n):
    B = idx.shape[0]
    flat = (np.arange(B)[:, None] * n + idx).ravel()
    return np.bincount(flat, minlength=B * n).reshape(B, n).astype(np.float64)

def mean_w(W, x):
    return (W @ x) / W.sum(axis=1)

def pct_ci(v):
    v = v[np.isfinite(v)]
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if len(v) else (float('nan'), float('nan'))

print('Krishna metrics and bootstrap helpers ready')


In [ ]:
t0 = time.time()
arrays = {}   # (ds, variant, source) -> (n, F, 5)
y_eval = {}
log = []
for ds in DATASETS:
    sv_dir = Path(REPO) / 'shap_values' / ds
    eval_idx = np.load(sv_dir / 'canonical_eval_idx.npy')
    y_eval[ds] = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')[eval_idx]
    for variant in MODEL_VARIANTS:
        for src, (arch, suffix) in SOURCES.items():
            model_name = f'{arch}_{variant}'
            if suffix == 'shap_shared':
                saved_idx = np.load(sv_dir / f'{model_name}_eval_idx_shared.npy')
                assert np.array_equal(saved_idx, eval_idx), f'{ds}/{model_name}: eval_idx_shared is not the canonical set'
                path, cached = sv_dir / f'{model_name}_shap_shared.npy', True
            else:
                t1 = time.time()
                path, cached = ensure_dnn_logit_shap(ds, model_name)
                log.append({'dataset': ds, 'model': model_name, 'cached': cached, 'seconds': round(time.time() - t1, 1)})
            arr = np.load(path)
            assert arr.shape[0] == len(eval_idx) and arr.shape[2] == 5, (ds, model_name, arr.shape)
            arrays[(ds, variant, src)] = arr
        print(f'{ds:15s} {variant:13s} ' + ' '.join(f'{s}={arrays[(ds, variant, s)].shape}' for s in SOURCES))
pd.DataFrame(log).to_csv(TABLES / f'{PREFIX}_shap_log.csv', index=False)
print(f'{len(arrays)} arrays; {(time.time() - t0) / 60:.1f} min')


In [ ]:
t0 = time.time()
rows = []
for ds in DATASETS:
    for variant in MODEL_VARIANTS:
        n = arrays[(ds, variant, 'rf_prob')].shape[0]
        for label, sa, sb, note in CONFIGS:
            A, B_ = arrays[(ds, variant, sa)], arrays[(ds, variant, sb)]
            for direction, (X, Y) in {'A->B': (A, B_), 'B->A': (B_, A)}.items():
                for i in range(n):
                    va, vb = aggregate_vector(X, i), aggregate_vector(Y, i)
                    for k in K_VALUES:
                        rows.append({'dataset': ds, 'variant': variant, 'config': label, 'space_note': note,
                                     'direction': direction, 'k': k, 'sample_position': i,
                                     'true_class': int(y_eval[ds][i]),
                                     **{m: fn(va, vb, k) for m, fn in METRICS.items()}})
        print(f'{ds:15s} {variant:13s} done  {(time.time() - t0) / 60:.1f} min')
df_ps = pd.DataFrame(rows)
df_ps.to_csv(TABLES / f'{PREFIX}_per_sample.csv', index=False)
print(f'{len(df_ps)} per-sample rows (expect {len(DATASETS) * len(MODEL_VARIANTS) * len(CONFIGS) * 2 * len(K_VALUES) * 1000})')

# Consistency with 06 on the original configs (A->B, K=10): the committed per-cell CIs
old = TABLES / 'bootstrap_cis_krishna_per_cell.csv'
if old.exists():
    o = pd.read_csv(old)
    o['config'] = o.pair + ' original'
    m = (df_ps[(df_ps.direction == 'A->B') & (df_ps.k == K_HEADLINE) & df_ps.config.str.endswith('original')]
         .groupby(['dataset', 'variant', 'config']).RC.mean().reset_index()
         .merge(o[['dataset', 'variant', 'config', 'mean_sample_RC']], on=['dataset', 'variant', 'config']))
    m['diff'] = (m.RC - m.mean_sample_RC).abs()
    print('\nvs bootstrap_cis_krishna_per_cell.csv (K=10):')
    print(m.round(4).to_string(index=False))


In [ ]:
t0 = time.time()
cell_rows, pooled_rows = [], []
n = 1000
W = boot_weights(boot_index(n), n)          # one draw of canonical positions, shared by every config and variant
Wp = np.hstack([W, W])                      # pooled over the two variants: same positions carried for both (cluster bootstrap)

def series(ds, variant, config, direction, k, metric):
    g = df_ps[(df_ps.dataset == ds) & (df_ps.variant == variant) & (df_ps.config == config) &
              (df_ps.direction == direction) & (df_ps.k == k)].sort_values('sample_position')
    return g[metric].values.astype(np.float64)

for ds in DATASETS:
    for direction in ['A->B', 'B->A']:
        for k in K_VALUES:
            base = {v: {m: series(ds, v, 'xgb-dnn original', direction, k, m) for m in METRICS} for v in MODEL_VARIANTS}
            for label, sa, sb, note in CONFIGS:
                per_variant = {}
                for variant in MODEL_VARIANTS:
                    x = {m: series(ds, variant, label, direction, k, m) for m in METRICS}
                    per_variant[variant] = x
                    row = {'dataset': ds, 'variant': variant, 'config': label, 'space_note': note,
                           'direction': direction, 'k': k, 'n': n}
                    for m in METRICS:
                        b = mean_w(W, x[m]); lo, hi = pct_ci(b)
                        row.update({m: float(x[m].mean()), f'{m}_lo': lo, f'{m}_hi': hi})
                    if label.startswith('xgb-dnn') and label != 'xgb-dnn original':
                        d = mean_w(W, x['RC'] - base[variant]['RC']); lo, hi = pct_ci(d)
                        row.update(delta_RC_vs_original=float((x['RC'] - base[variant]['RC']).mean()),
                                   delta_RC_lo=lo, delta_RC_hi=hi)
                    cell_rows.append(row)
                # pooled over variants, cluster bootstrap on canonical positions
                row = {'dataset': ds, 'variant': 'POOLED', 'config': label, 'space_note': note,
                       'direction': direction, 'k': k, 'n': 2 * n}
                for m in METRICS:
                    xp = np.concatenate([per_variant[v][m] for v in MODEL_VARIANTS])
                    b = mean_w(Wp, xp); lo, hi = pct_ci(b)
                    row.update({m: float(xp.mean()), f'{m}_lo': lo, f'{m}_hi': hi})
                if label.startswith('xgb-dnn') and label != 'xgb-dnn original':
                    dp = np.concatenate([per_variant[v]['RC'] - base[v]['RC'] for v in MODEL_VARIANTS])
                    d = mean_w(Wp, dp); lo, hi = pct_ci(d)
                    row.update(delta_RC_vs_original=float(dp.mean()), delta_RC_lo=lo, delta_RC_hi=hi)
                pooled_rows.append(row)

df_cells = pd.DataFrame(cell_rows + pooled_rows)
def sig(lo, hi):
    if not np.isfinite(lo): return 'n/a'
    if lo > 0: return 'POS'
    if hi < 0: return 'NEG'
    return 'ns'
df_cells['RC_sig'] = [sig(a, b) for a, b in zip(df_cells.RC_lo, df_cells.RC_hi)]
df_cells.to_csv(TABLES / f'{PREFIX}_cells.csv', index=False)
print(f'{len(df_cells)} rows; {(time.time() - t0) / 60:.1f} min')

pd.set_option('display.width', 250)
head = df_cells[(df_cells.k == K_HEADLINE) & (df_cells.direction == 'A->B')]
print(f'\nK={K_HEADLINE}, direction A->B (top-k from the first-named model)')
print(head[['dataset', 'variant', 'config', 'RC', 'RC_lo', 'RC_hi', 'RC_sig', 'FA', 'PRA',
            'delta_RC_vs_original', 'delta_RC_lo', 'delta_RC_hi']].round(3).to_string(index=False))
head_r = df_cells[(df_cells.k == K_HEADLINE) & (df_cells.direction == 'B->A') & (df_cells.variant == 'POOLED')]
print(f'\nK={K_HEADLINE}, direction B->A, pooled')
print(head_r[['dataset', 'config', 'RC', 'RC_lo', 'RC_hi', 'RC_sig', 'FA', 'PRA']].round(3).to_string(index=False))


In [ ]:
def fmt(x, d=3):
    return 'n/a' if x is None or (isinstance(x, float) and not np.isfinite(x)) else f'{x:.{d}f}'

def md_table(frame, cols, digits=3):
    head = '| ' + ' | '.join(cols) + ' |\n|' + '|'.join(['---'] * len(cols)) + '|\n'
    body = ''
    for _, r in frame[cols].iterrows():
        body += '| ' + ' | '.join(fmt(v, digits) if isinstance(v, (float, np.floating)) else str(v) for v in r.values) + ' |\n'
    return head + body

pooled10 = df_cells[(df_cells.k == K_HEADLINE) & (df_cells.variant == 'POOLED')]
summary = {
    'timestamp': datetime.now().isoformat(), 'notebook': '13_krishna_output_space.ipynb',
    'n_bootstrap': N_BOOTSTRAP, 'bootstrap_seed': BOOTSTRAP_SEED, 'k_headline': K_HEADLINE,
    'sources': {k: list(v) for k, v in SOURCES.items()}, 'configs': [c[0] for c in CONFIGS],
    'pooled_k10': pooled10.to_dict(orient='records'),
}
with open(TABLES / f'{PREFIX}_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))

lines = []
lines.append('# Krishna cross-architecture agreement under matched SHAP output spaces\n')
lines.append(f'Generated by notebooks/13_krishna_output_space.ipynb on {datetime.now():%Y-%m-%d}. '
             f'Metrics as in 06 (aggregate |SHAP| summed over classes, K = {K_HEADLINE} headline, K in {K_VALUES}). '
             f'B = {N_BOOTSTRAP} percentile bootstrap over canonical positions, seed {BOOTSTRAP_SEED}; pooled rows carry both variants per position.\n')
lines.append('## 1. Output spaces\n')
lines.append('- rf: sklearn TreeExplainer raw output is predict_proba (probability space).\n'
             '- xgb: XGBoost TreeExplainer raw output is the per-class margin (logit space). TreeExplainer cannot map multiclass softmax to probability space, so the matched comparison is made in logit space.\n'
             '- dnn_prob: GradientExplainer on the softmax head (04c). dnn_logit: GradientExplainer on the logit head, same background, same eval set, same seed (this notebook).\n')
lines.append(f'\n## 2. Pooled RC at K={K_HEADLINE}, top-k taken from the first-named model\n')
lines.append(md_table(pooled10[pooled10.direction == 'A->B'].sort_values(['dataset', 'config']),
                      ['dataset', 'config', 'space_note', 'RC', 'RC_lo', 'RC_hi', 'RC_sig', 'FA', 'PRA',
                       'delta_RC_vs_original', 'delta_RC_lo', 'delta_RC_hi']))
lines.append(f'\n## 3. Pooled RC at K={K_HEADLINE}, top-k taken from the second-named model\n')
lines.append(md_table(pooled10[pooled10.direction == 'B->A'].sort_values(['dataset', 'config']),
                      ['dataset', 'config', 'RC', 'RC_lo', 'RC_hi', 'RC_sig', 'FA', 'PRA']))
lines.append('\n## 4. Per-variant cells, all K, direction A->B\n')
lines.append(md_table(df_cells[(df_cells.direction == 'A->B') & (df_cells.variant != 'POOLED')].sort_values(['dataset', 'config', 'variant', 'k']),
                      ['dataset', 'variant', 'config', 'k', 'RC', 'RC_lo', 'RC_hi', 'RC_sig', 'FA', 'RA', 'PRA']))
with open(DOCS / f'{PREFIX}_findings.md', 'w') as f:
    f.write('\n'.join(lines))
print(f'saved docs/{PREFIX}_findings.md and results/tables/{PREFIX}_summary.json')


In [ ]:
os.chdir(REPO)
!git config user.name "Md Anas Biswas"
!git config user.email "anasbiswas@gmail.com"

import nbformat as _nbf
_nb_path = Path(REPO) / 'notebooks' / '13_krishna_output_space.ipynb'
if _nb_path.exists():
    _nb = _nbf.read(_nb_path, 4)
    for _c in _nb.cells:
        if _c.cell_type == 'code':
            _c.outputs, _c.execution_count = [], None
    _nbf.write(_nb, _nb_path)
    print(f'outputs stripped: {_nb_path.relative_to(REPO)}')
else:
    raise FileNotFoundError(f'{_nb_path} not found: save this notebook under notebooks/ before committing')

!git add notebooks/13_krishna_output_space.ipynb
!git add results/tables/krishna_space_*.csv results/tables/krishna_space_summary.json
!git add docs/krishna_space_findings.md
!git add shap_values/*/dnn_5class_*_shap_shared_logit_meta.json
!git status --short | head -40
!git commit -m "Notebook 13: Krishna agreement recomputed with DNN SHAP in logit space to match XGBoost margins; both rank directions; K=5/10/15; cluster-bootstrap CIs; paired delta against the 06 configuration"
!git push origin main
!git log --oneline -3
